# QC Summary: Post-CellBender + Filtering

Visualize QC metrics after CellBender correction and MAD-based filtering.

**Contents:**
1. Per-sample violin plots of QC metrics (pre/post filtering)
2. Scatter plots: total_counts vs n_genes colored by mito%
3. Summary table: cells retained per sample
4. CellBender correction validation (marker gene expression)
5. Scrublet doublet score distributions

In [ ]:
import sys
sys.path.insert(0, '../pipeline')

import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from utils import load_config, set_plotting_defaults
set_plotting_defaults()

cfg = load_config('../pipeline/config.yaml')

In [ ]:
# Load QC summary
qc_summary = pd.read_csv(Path(cfg['paths']['qc_output']) / 'qc_summary.csv')
print(f"Samples processed: {len(qc_summary)}")
print(f"Total cells retained: {qc_summary['cells_final'].sum():,}")
print(f"Median retention rate: {qc_summary['pct_retained'].median():.1f}%")
print()
qc_summary.sort_values('pct_retained')

In [ ]:
# Load a few QC'd samples and plot metrics
qc_dir = Path(cfg['paths']['qc_output'])
h5ad_files = sorted(qc_dir.glob('*_qc.h5ad'))

# Load all into a single AnnData for plotting
adata_list = []
for f in h5ad_files:
    adata = sc.read_h5ad(str(f))
    adata_list.append(adata)

if adata_list:
    import anndata as ad
    adata_all = ad.concat(adata_list, join='outer', label='sample_id',
                          keys=[f.stem.replace('_qc', '') for f in h5ad_files])
    print(f"Combined QC data: {adata_all.n_obs} cells x {adata_all.n_vars} genes")
else:
    print("No QC'd h5ad files found. Run pipeline/02_qc_filtering.py first.")
    adata_all = None

In [ ]:
# Violin plots of QC metrics
if adata_all is not None:
    metrics = ['n_genes_by_counts', 'total_counts', 'pct_counts_mt']
    available = [m for m in metrics if m in adata_all.obs.columns]
    
    fig, axes = plt.subplots(len(available), 1,
                             figsize=(max(14, adata_all.obs['sample_id'].nunique() * 0.5),
                                      4 * len(available)))
    if len(available) == 1:
        axes = [axes]
    
    for ax, metric in zip(axes, available):
        sc.pl.violin(adata_all, metric, groupby='sample_id',
                     rotation=90, ax=ax, show=False, stripplot=False)
        ax.set_title(f'{metric} (post-QC)')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Scatter: total_counts vs n_genes, colored by mito%
if adata_all is not None and 'pct_counts_mt' in adata_all.obs.columns:
    n_samples = min(6, adata_all.obs['sample_id'].nunique())
    samples = adata_all.obs['sample_id'].value_counts().head(n_samples).index
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    for i, sample in enumerate(samples):
        if i >= len(axes):
            break
        mask = adata_all.obs['sample_id'] == sample
        sub = adata_all.obs[mask]
        sc = axes[i].scatter(sub['total_counts'], sub['n_genes_by_counts'],
                             c=sub['pct_counts_mt'], cmap='RdYlBu_r',
                             s=3, alpha=0.5, vmin=0, vmax=20)
        axes[i].set_xlabel('Total counts')
        axes[i].set_ylabel('N genes')
        axes[i].set_title(f'{sample} (n={mask.sum()})')
        plt.colorbar(sc, ax=axes[i], label='% mito')
    
    for i in range(len(samples), len(axes)):
        axes[i].set_visible(False)
    
    plt.suptitle('Total counts vs N genes (colored by mito%)', y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
# Doublet score distributions
if adata_all is not None and 'doublet_score' in adata_all.obs.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    adata_all.obs['doublet_score'].hist(bins=50, ax=ax, alpha=0.7)
    ax.set_xlabel('Scrublet doublet score')
    ax.set_ylabel('Number of cells')
    ax.set_title('Doublet score distribution (all samples, post-filtering)')
    plt.tight_layout()
    plt.show()
    
    print(f"Mean doublet score: {adata_all.obs['doublet_score'].mean():.4f}")
    print(f"Cells with score > 0.25: {(adata_all.obs['doublet_score'] > 0.25).sum()}")

In [ ]:
# Samples losing >50% cells
low_retention = qc_summary[qc_summary['pct_retained'] < 50]
if len(low_retention) > 0:
    print("WARNING: Samples with >50% cell loss:")
    print(low_retention[['sample', 'cells_raw', 'cells_final', 'pct_retained']].to_string())
else:
    print("All samples retained >50% of cells.")